In [1]:
import os
import sys
import glob
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. DYNAMIC PROJECT ROOT DETECTION
# ==============================================================================
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Analysis").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing 'Analysis' and 'results'. "
        "Run this notebook from the project directory or one of its subdirectories."
    )

PROJECT_ROOT = find_project_root()
ANALYSIS_DIR = PROJECT_ROOT / "Analysis"
RESULTS_DIR = PROJECT_ROOT / "results"

if not any(RESULTS_DIR.glob("*.csv")):
    raise FileNotFoundError(f"No result CSV files found under {RESULTS_DIR}")

print(f"🚀 Project root detected: {PROJECT_ROOT}")
print(f"📁 Analysis Dir: {ANALYSIS_DIR}")
print(f"📁 Results Dir:  {RESULTS_DIR}")

# ==============================================================================
# 2. STATISTICAL UTILITY FUNCTIONS
# ==============================================================================
def compute_cohens_d(x, y):
    """Computes Cohen's d for paired samples: d = mean(x - y) / std(x - y)"""
    diff = np.asarray(x) - np.asarray(y)
    n = len(diff)
    if n < 2:
        return 0.0
    return np.mean(diff) / (np.std(diff, ddof=1) + 1e-9)

def compute_bootstrap_ci(data, n_bootstraps=2000, ci=95, seed=42):
    """Computes non-parametric bootstrap confidence interval."""
    rng = np.random.default_rng(seed)
    data = np.asarray(data)
    boot_means = [np.mean(rng.choice(data, size=len(data), replace=True)) for _ in range(n_bootstraps)]
    lower = np.percentile(boot_means, (100 - ci) / 2)
    upper = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return lower, upper

def fdr_correction_bh(p_values):
    """Applies Benjamini-Hochberg FDR correction across multiple comparisons."""
    p_vals = np.asarray(p_values)
    n = len(p_vals)
    sorted_indices = np.argsort(p_vals)
    sorted_p = p_vals[sorted_indices]
    
    adjusted = np.zeros(n)
    cummin = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        adj_p = (sorted_p[i] * n) / rank
        cummin = min(cummin, adj_p)
        adjusted[i] = cummin
        
    rev_indices = np.argsort(sorted_indices)
    return np.clip(adjusted[rev_indices], 0.0, 1.0)

# ==============================================================================
# 3. LOAD & VALIDATE ALL 9 FOLD-RESULT CSVS
# ==============================================================================
MODELS = ['eegnet', 'cvt', 'vit_pe']
DATASETS = ['Raw', 'Filtered_8_30Hz', 'Filtered_8_13Hz']
METRICS = ['accuracy', 'f1', 'precision', 'recall', 'roc_auc']

data_store = {}

for model in MODELS:
    for ds in DATASETS:
        filename = f"{model}_results_{ds}.csv"
        filepath = RESULTS_DIR / filename
        if not filepath.exists():
            raise FileNotFoundError(f"Missing required file: {filepath}")
        
        df = pd.read_csv(filepath)
        df.columns = [c.strip().lower() for c in df.columns]
        
        # Validate columns
        for m in METRICS + ['fold']:
            if m not in df.columns:
                raise ValueError(f"File {filename} is missing required column: {m}")
                
        data_store[(model, ds)] = df.sort_values(by='fold').reset_index(drop=True)

print("✅ All 9 fold result CSV files loaded and validated successfully!")

# ==============================================================================
# 4. SUMMARY PERFORMANCE TABLE WITH 95% BOOTSTRAP CI
# ==============================================================================
summary_records = []

for ds in DATASETS:
    for model in MODELS:
        df = data_store[(model, ds)]
        rec = {
            'Dataset': ds,
            'Model': model.upper().replace('_', '-'),
            'N_Folds': len(df)
        }
        for metric in METRICS:
            vals = df[metric].values * 100 # percentage scale
            mean_val = np.mean(vals)
            std_val = np.std(vals, ddof=1)
            ci_low, ci_high = compute_bootstrap_ci(vals)
            
            rec[f"{metric.upper()}_Mean"] = round(mean_val, 2)
            rec[f"{metric.upper()}_SD"] = round(std_val, 2)
            rec[f"{metric.upper()}_CI95_Low"] = round(ci_low, 2)
            rec[f"{metric.upper()}_CI95_High"] = round(ci_high, 2)
            rec[f"{metric.upper()} (Mean±SD)"] = f"{mean_val:.2f} ± {std_val:.2f}%"
            rec[f"{metric.upper()} [95% CI]"] = f"[{ci_low:.2f}, {ci_high:.2f}]%"
            
        summary_records.append(rec)

summary_df = pd.DataFrame(summary_records)
summary_output_path = ANALYSIS_DIR / "summary_fold_performance.csv"
summary_df.to_csv(summary_output_path, index=False)
print(f"✅ Saved summary metrics with 95% Bootstrap CI to: {summary_output_path.name}")

print("\n--- SUMMARY PERFORMANCE TABLE (Across 5 Folds) ---")
display_cols = ['Dataset', 'Model'] + [f"{m.upper()} (Mean±SD)" for m in METRICS]
print(summary_df[display_cols].to_string(index=False))

# ==============================================================================
# 5. PAIRWISE STATISTICAL COMPARISONS & FDR CORRECTION
# ==============================================================================
stat_comparisons = []
PAIRS = [
    ('cvt', 'eegnet', 'CvT vs EEGNet'),
    ('vit_pe', 'eegnet', 'ViT-PE vs EEGNet'),
    ('vit_pe', 'cvt', 'ViT-PE vs CvT')
]

for ds in DATASETS:
    for m1, m2, label in PAIRS:
        df1 = data_store[(m1, ds)]
        df2 = data_store[(m2, ds)]
        
        for metric in METRICS:
            scores1 = df1[metric].values
            scores2 = df2[metric].values
            diff = (scores1 - scores2) * 100 # percentage scale
            
            # 1. Paired Student's t-test
            t_stat, p_val_ttest = stats.ttest_rel(scores1, scores2)
            
            # 2. Wilcoxon signed-rank test
            try:
                w_stat, p_val_wilcoxon = stats.wilcoxon(scores1, scores2)
            except Exception:
                p_val_wilcoxon = 1.0
                w_stat = 0.0
                
            # 3. Cohen's d (Effect Size)
            d = compute_cohens_d(scores1, scores2)
            if abs(d) >= 0.8:
                d_interp = "Large"
            elif abs(d) >= 0.5:
                d_interp = "Medium"
            elif abs(d) >= 0.2:
                d_interp = "Small"
            else:
                d_interp = "Negligible"
                
            # 4. Bootstrap CI for Difference
            ci_diff_low, ci_diff_high = compute_bootstrap_ci(diff)
            
            stat_comparisons.append({
                'Dataset': ds,
                'Comparison': label,
                'Model_1': m1,
                'Model_2': m2,
                'Metric': metric.upper(),
                'Mean_Diff (%)': round(np.mean(diff), 2),
                'CI95_Diff_Low (%)': round(ci_diff_low, 2),
                'CI95_Diff_High (%)': round(ci_diff_high, 2),
                'CI95_Diff': f"[{ci_diff_low:.2f}, {ci_diff_high:.2f}]%",
                't_stat': round(t_stat, 4),
                'raw_p_ttest': p_val_ttest,
                'p_val (Paired t-test)': f"{p_val_ttest:.4e}" if p_val_ttest < 0.001 else f"{p_val_ttest:.4f}",
                'raw_p_wilcoxon': p_val_wilcoxon,
                'p_val (Wilcoxon)': f"{p_val_wilcoxon:.4e}" if p_val_wilcoxon < 0.001 else f"{p_val_wilcoxon:.4f}",
                "Cohen's d": round(d, 4),
                'Effect Size': d_interp
            })

stat_df = pd.DataFrame(stat_comparisons)

# Apply Benjamini-Hochberg FDR correction across all comparisons
stat_df['p_val_fdr_bh'] = fdr_correction_bh(stat_df['raw_p_ttest'].values)
stat_df['p_val_fdr_bh_str'] = stat_df['p_val_fdr_bh'].apply(
    lambda p: f"{p:.4e}" if p < 0.001 else f"{p:.4f}"
)
stat_df['Significant (FDR q<0.05)'] = stat_df['p_val_fdr_bh'].apply(lambda p: 'Yes' if p < 0.05 else 'No')

# Save updated statistical results
stat_output_path = ANALYSIS_DIR / "statistical_significance_results.csv"
stat_df.to_csv(stat_output_path, index=False)
print(f"✅ Saved updated statistical significance results (with FDR) to: {stat_output_path.name}")

# ==============================================================================
# 6. PRINT SUMMARY TABLE FOR REVIEWER RESPONSE (ACCURACY & F1)
# ==============================================================================
key_cols = [
    'Dataset', 'Comparison', 'Metric', 'Mean_Diff (%)', 'CI95_Diff',
    'p_val (Paired t-test)', 'p_val_fdr_bh_str', "Cohen's d", 'Effect Size', 'Significant (FDR q<0.05)'
]
key_stats = stat_df[stat_df['Metric'].isin(['ACCURACY', 'F1'])][key_cols]
key_stats.rename(columns={'p_val_fdr_bh_str': 'p_val (FDR-BH)'}, inplace=True)

print("\n" + "="*110)
print("🎯 STATISTICAL COMPARISON SUMMARY TABLE (WITH 95% BOOTSTRAP CI & BENJAMINI-HOCHBERG FDR)")
print("="*110)
print(key_stats.to_string(index=False))

🚀 Project root detected: C:\Users\ASUS\Anaconda Projects\Article
📁 Analysis Dir: C:\Users\ASUS\Anaconda Projects\Article\Analysis
📁 Results Dir:  C:\Users\ASUS\Anaconda Projects\Article\results
✅ All 9 fold result CSV files loaded and validated successfully!
✅ Saved summary metrics with 95% Bootstrap CI to: summary_fold_performance.csv

--- SUMMARY PERFORMANCE TABLE (Across 5 Folds) ---
        Dataset  Model ACCURACY (Mean±SD)  F1 (Mean±SD) PRECISION (Mean±SD) RECALL (Mean±SD) ROC_AUC (Mean±SD)
            Raw EEGNET      86.26 ± 4.99% 86.17 ± 5.61%       85.76 ± 2.90%    86.82 ± 8.77%     93.42 ± 4.33%
            Raw    CVT      76.23 ± 3.69% 77.22 ± 2.53%       74.87 ± 6.20%    80.31 ± 4.93%     85.11 ± 4.00%
            Raw VIT-PE      67.39 ± 3.42% 67.51 ± 2.54%       67.53 ± 4.30%    67.60 ± 1.98%     73.99 ± 3.76%
Filtered_8_30Hz EEGNET      84.24 ± 4.12% 83.79 ± 4.32%       86.21 ± 4.26%    81.52 ± 4.65%     91.58 ± 4.46%
Filtered_8_30Hz    CVT      77.26 ± 2.25% 76.85 ± 2.35%